# 07 · Comparativa de los modelos de detección

**Objetivo:** comparar de forma rigurosa los cuatro modelos de detección entrenados (v1, v2, v3, v4) evaluándolos sobre el mismo conjunto de test independiente, midiendo tanto precisión como velocidad de inferencia.

**Contexto:** a lo largo de la fase 2 se han entrenado cuatro modelos variando una característica cada vez: tamaño del dataset (v1→v2), data augmentation (v2→v3) y resolución de entrada (v2→v4). Hasta ahora cada modelo se evaluó sobre su propio conjunto de validación, lo que no permite una comparación justa. Este notebook los evalúa a todos sobre el mismo conjunto de test, que ningún modelo ha visto durante el entrenamiento, garantizando una comparación metodológicamente correcta.

**Entrada:** los cuatro modelos en `models/` y el conjunto de test del dataset.

**Salida:** tabla comparativa de precisión y velocidad, y selección justificada del modelo final.

## 1. Configuración

In [1]:
import torch
from ultralytics import YOLO
from pathlib import Path
import time

REPO_ROOT = Path("../..").resolve()
MODELS_DIR = REPO_ROOT / "models"
DATASET_DIR = REPO_ROOT / "data" / "dataset"
DATA_YAML = DATASET_DIR / "data.yaml"

# Modelos a comparar (nombre descriptivo: archivo, resolución de evaluación)
MODELOS = {
    "v1 (152, 640px)":      ("best_v1.pt", 640),
    "v2 (743, 640px)":      ("best_v2.pt", 640),
    "v3 (743+aug, 640px)":  ("best_v3.pt", 640),
    "v4 (743, 1280px)":     ("best_v4.pt", 1280),
}

print(f"GPU: {torch.cuda.get_device_name(0)}")
print(f"Modelos a comparar: {len(MODELOS)}")
for nombre, (archivo, res) in MODELOS.items():
    existe = (MODELS_DIR / archivo).exists()
    print(f"  {nombre}: {archivo} {'OK' if existe else 'NO ENCONTRADO'}")

GPU: NVIDIA GeForce RTX 2060
Modelos a comparar: 4
  v1 (152, 640px): best_v1.pt OK
  v2 (743, 640px): best_v2.pt OK
  v3 (743+aug, 640px): best_v3.pt OK
  v4 (743, 1280px): best_v4.pt OK


## 2. Verificación del conjunto de test

La comparación se realiza sobre el conjunto de test, que ningún modelo ha visto durante el entrenamiento. Se actualiza el data.yaml para asegurar que la evaluación usa la partición de test.

In [2]:
test_imgs = list((DATASET_DIR / "test" / "images").glob("*.jpg"))
print(f"Imágenes en el conjunto de test: {len(test_imgs)}")

# Asegurar que el data.yaml incluye la línea de test
yaml_content = f"""train: {DATASET_DIR}/train/images
val: {DATASET_DIR}/valid/images
test: {DATASET_DIR}/test/images
nc: 3
names: ['ball', 'player', 'referee']
"""
with open(DATA_YAML, "w", encoding="utf-8") as f:
    f.write(yaml_content)
print("data.yaml actualizado con partición de test")

Imágenes en el conjunto de test: 74
data.yaml actualizado con partición de test


## 3. Evaluación de cada modelo sobre el test

Para cada modelo se calculan las métricas de precisión (mAP50 y mAP50-95 por clase) sobre el conjunto de test. Se usa `split='test'` para forzar la evaluación sobre esa partición.

In [3]:
resultados = {}

for nombre, (archivo, resolucion) in MODELOS.items():
    print(f"\nEvaluando {nombre}...")
    model = YOLO(str(MODELS_DIR / archivo))

    metrics = model.val(
        data=str(DATA_YAML),
        split="test",
        imgsz=resolucion,
        workers=0,
        verbose=False
    )

    resultados[nombre] = {
        "mAP50": metrics.box.map50,
        "mAP50-95": metrics.box.map,
        "ball_mAP50": metrics.box.ap50[0],
        "player_mAP50": metrics.box.ap50[1],
        "referee_mAP50": metrics.box.ap50[2],
        "speed_inference": metrics.speed["inference"],
    }
    print(f"  mAP50 global: {metrics.box.map50:.3f}")

print("\nEvaluación completada")


Evaluando v1 (152, 640px)...
Ultralytics 8.4.53  Python-3.13.13 torch-2.6.0+cu124 CUDA:0 (NVIDIA GeForce RTX 2060, 6144MiB)
Model summary (fused): 73 layers, 11,126,745 parameters, 0 gradients, 28.4 GFLOPs
val: Fast image access  (ping: 0.20.1 ms, read: 77.611.7 MB/s, size: 70.9 KB)
val: Scanning C:\Users\sgghu\OneDrive\Documents\GitHub\TFG\data\dataset\test\labels... 74 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 74/74 1.3Kit/s 0.1s
val: New cache created: C:\Users\sgghu\OneDrive\Documents\GitHub\TFG\data\dataset\test\labels.cache
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 5/5 1.8it/s 2.8s0.9s
                   all         74       1782      0.589      0.514      0.478      0.212
Speed: 1.4ms preprocess, 12.7ms inference, 0.0ms loss, 1.5ms postprocess per image
Results saved to C:\Users\sgghu\OneDrive\Documents\GitHub\TFG\notebooks\fase2_deteccion_jugadores\runs\detect\val-7
  mAP50 global: 0.478

Evaluando v2 

## 4. Tabla comparativa de precisión

Se presenta el mAP50 de cada clase y el promedio, para todos los modelos, sobre el conjunto de test.

In [4]:
print(f"{'Modelo':<22} {'Balón':>8} {'Jugador':>8} {'Árbitro':>8} {'mAP50':>8} {'mAP50-95':>9}")
print("-" * 70)

for nombre, r in resultados.items():
    print(f"{nombre:<22} {r['ball_mAP50']:>8.3f} {r['player_mAP50']:>8.3f} "
          f"{r['referee_mAP50']:>8.3f} {r['mAP50']:>8.3f} {r['mAP50-95']:>9.3f}")

Modelo                    Balón  Jugador  Árbitro    mAP50  mAP50-95
----------------------------------------------------------------------
v1 (152, 640px)           0.398    0.893    0.143    0.478     0.212
v2 (743, 640px)           0.445    0.949    0.516    0.637     0.350
v3 (743+aug, 640px)       0.466    0.929    0.566    0.654     0.350
v4 (743, 1280px)          0.574    0.962    0.581    0.706     0.385


## 5. Tabla comparativa de velocidad

La velocidad de inferencia es un factor clave para procesar vídeo. Se compara el tiempo por imagen y los fotogramas por segundo equivalentes de cada modelo.

In [5]:
print(f"{'Modelo':<22} {'ms/imagen':>12} {'FPS aprox.':>12}")
print("-" * 48)

for nombre, r in resultados.items():
    ms = r["speed_inference"]
    fps = 1000 / ms if ms > 0 else 0
    print(f"{nombre:<22} {ms:>12.1f} {fps:>12.0f}")

Modelo                    ms/imagen   FPS aprox.
------------------------------------------------
v1 (152, 640px)                12.7           79
v2 (743, 640px)                11.9           84
v3 (743+aug, 640px)            17.4           57
v4 (743, 1280px)               31.8           31


## 6. Visualización comparativa

In [6]:
import matplotlib.pyplot as plt
import numpy as np

nombres = list(resultados.keys())
ball = [resultados[n]["ball_mAP50"] for n in nombres]
player = [resultados[n]["player_mAP50"] for n in nombres]
referee = [resultados[n]["referee_mAP50"] for n in nombres]

x = np.arange(len(nombres))
width = 0.25

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))

# Precisión por clase
ax1.bar(x - width, ball, width, label="Balón", color="#f1c40f")
ax1.bar(x, player, width, label="Jugador", color="#2ecc71")
ax1.bar(x + width, referee, width, label="Árbitro", color="#e74c3c")
ax1.set_ylabel("mAP50")
ax1.set_title("Precisión por clase y modelo (test)")
ax1.set_xticks(x)
ax1.set_xticklabels(nombres, rotation=15, ha="right", fontsize=9)
ax1.legend()
ax1.set_ylim(0, 1)
ax1.grid(axis="y", alpha=0.3)

# Velocidad
fps = [1000 / resultados[n]["speed_inference"] for n in nombres]
ax2.bar(x, fps, color="#3498db")
ax2.set_ylabel("FPS aproximados")
ax2.set_title("Velocidad de inferencia por modelo")
ax2.set_xticks(x)
ax2.set_xticklabels(nombres, rotation=15, ha="right", fontsize=9)
ax2.grid(axis="y", alpha=0.3)

plt.tight_layout()
plt.show()

<Figure size 1600x600 with 2 Axes>

## 7. Conclusiones y selección del modelo final

La evaluación sobre el conjunto de test (74 imágenes que ningún modelo vio durante el entrenamiento) confirma la evolución del detector a lo largo de la fase 2. Las métricas sobre test son algo más bajas que las de validación, lo cual es esperable y deseable: reflejan el rendimiento real del modelo sobre datos completamente nuevos.

**Efecto del tamaño del dataset (v1 → v2):** el aumento de 152 a 743 imágenes mediante active learning mejora drásticamente el rendimiento general (mAP50 de 0.478 a 0.637). La mejora es especialmente notable en el árbitro (de 0.143 a 0.516), la clase con menos ejemplos, que se beneficia mucho de más datos.

**Efecto del data augmentation (v2 → v3):** las augmentaciones de color mejoran ligeramente el árbitro (0.516 a 0.566) y el balón (0.445 a 0.466), confirmando que aportan robustez frente a la variabilidad cromática. La mejora del mAP50 global es modesta (0.637 a 0.654), pero el coste en velocidad es apreciable (de 84 a 57 FPS), debido a que el modelo aprende de un dataset más variado.

**Efecto de la resolución (v2 → v4):** el aumento a 1280px produce la mayor mejora del balón de todo el proyecto (0.445 a 0.574 sobre test), elevando el mAP50 global a 0.706, el mejor de los cuatro modelos. El coste es una inferencia 2.5 veces más lenta (de 84 a 31 FPS).

**Selección del modelo final:** el modelo v4 (1280px) se selecciona como detector de referencia por ofrecer la mejor precisión en todas las clases simultáneamente, especialmente en el balón, que era el punto débil del sistema. Su velocidad de 31 FPS es suficiente para el procesamiento de vídeo offline previsto en el sistema, ya que no se requiere tiempo real.

**Tabla resumen sobre test:**

| Modelo | Balón | Jugador | Árbitro | mAP50 | FPS |
|---|---|---|---|---|---|
| v1 (152, 640px) | 0.398 | 0.893 | 0.143 | 0.478 | 79 |
| v2 (743, 640px) | 0.445 | 0.949 | 0.516 | 0.637 | 84 |
| v3 (743+aug, 640px) | 0.466 | 0.929 | 0.566 | 0.654 | 57 |
| **v4 (743, 1280px)** | **0.574** | **0.962** | **0.581** | **0.706** | 31 |

**Vía de mejora futura:** combinar la alta resolución del v4 con el data augmentation del v3, y aplicar técnicas data-centric (imágenes de fondo vacío para reducir falsos positivos, y más ejemplos reales de balón mediante una segunda iteración de active learning) constituiría un modelo v5 con potencial para superar al v4.